# SASRec. Квантование с обучением (QAT) и оценка на CPU

In [1]:
cd /mnt/zkmartirosian/del/SasRec/

/mnt/zkmartirosian/del/SasRec


За основу возьмем модель из оригинального репозитория https://github.com/pmixer/SASRec.pytorch/blob/main/python/model.py
Данные для обучения возьмем из того же репозитория https://github.com/pmixer/SASRec.pytorch/blob/main/python/data/ml-1m.txt

In [2]:
# !mkdir data
# !wget https://raw.githubusercontent.com/pmixer/SASRec.pytorch/refs/heads/main/python/data/ml-1m.txt
# !mv ml-1m.txt data/.

In [3]:
import os
import torch

from tqdm import tqdm
from torch import nn
from pathlib import Path
from typing import Any, Dict
from train_functions import train_fp32, train_qat, apply_adaround

from models.quantization import QuantSASRec
from data.dataloader import create_dataloaders
from utils import load_config, set_random_seeds, ensure_dir

In [4]:
class Args:
    """Helper to pass config parameters to SASRec model."""
    def __init__(self, config: Dict[str, Any]):
        self.hidden_units = config["model"]["hidden_units"]
        self.num_blocks = config["model"]["num_blocks"]
        self.num_heads = config["model"]["num_heads"]
        self.dropout_rate = config["model"]["dropout_rate"]
        self.maxlen = config["model"]["maxlen"]
        self.device = torch.device(config["experiment"].get("device", "cuda" if torch.cuda.is_available() else "cpu"))
        self.norm_first = config["model"].get("norm_first", False)

In [5]:
def start_train(config_name):
    config_path = os.path.join("./configs", config_name)
    config = load_config(config_path)

    set_random_seeds(config["experiment"].get("seed", 42))
    args = Args(config)
    
    train_loader, _, _, dataset = create_dataloaders(
            config, seed=config["experiment"].get("seed", 42))
    
    _, _, _, usernum, itemnum = dataset
    model = QuantSASRec(usernum, itemnum, args).to(args.device)
    
    quant_cfg = config.get("quantization", {})
    strategy_name = quant_cfg.get("method", "fp32").lower()
    strategy_name = quant_cfg.get("method", "fp32").lower()
    checkpoint_dir = Path(config.get("paths", {}).get("checkpoints_dir", "./checkpoints"))
    run_name = config["experiment"].get("run_name", "sasrec_run")
    checkpoint_subdir = checkpoint_dir / run_name
    ensure_dir(checkpoint_subdir)
    results_dir = Path(config.get("paths", {}).get("results_dir", "./results"))
    ensure_dir(results_dir)

    logger = None
    criterion = nn.BCEWithLogitsLoss()

    if strategy_name == "fp32":
        train_fp32(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            config=config,
            criterion=criterion,
            args=args,
            checkpoint_dir=str(checkpoint_subdir),
            save_name="sasrec_fp32.pth",
            logger=logger
        )  
    elif strategy_name in ["lsq", "apot", "qdrop"]:
        train_qat(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            config=config,
            criterion=criterion,
            args=args,
            strategy_name=strategy_name,
            quant_config=quant_cfg,
            checkpoint_dir=str(checkpoint_subdir),
            save_name=f"sasrec_{strategy_name}.pth",
            logger=logger
        )
    elif strategy_name == "adaround":
        fp32_epochs = config["training"].get("fp32_epochs", 0)
        fp32_ckpt = quant_cfg.get("base_checkpoint", None)
        
        fp32_ckpt_name = "sasrec_fp32_for_adaround.pth"
        
        if fp32_epochs > 0:
             train_fp32(
                model=model,
                train_loader=train_loader,
                dataset=dataset,
                config=config,
                criterion=criterion,
                args=args,
                checkpoint_dir=str(checkpoint_subdir),
                save_name=fp32_ckpt_name,
                logger=logger
            )
        elif fp32_ckpt:
             fp32_ckpt_path = Path(fp32_ckpt)
             if not fp32_ckpt_path.exists():
                 fp32_ckpt_path = checkpoint_subdir / fp32_ckpt
             
             if not fp32_ckpt_path.exists():
                  raise FileNotFoundError(f"Base checkpoint not found: {fp32_ckpt}")
             if fp32_ckpt_path.parent == checkpoint_subdir:
                 fp32_ckpt_name = fp32_ckpt_path.name
             else:
                 fp32_ckpt_name = str(fp32_ckpt_path)
        
        apply_adaround(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            args=args,
            fp32_checkpoint=fp32_ckpt_name,
            adaround_config=quant_cfg,
            checkpoint_dir=str(checkpoint_subdir),
            save_name="sasrec_adaround.pth",
            logger=logger,
            config=config,
        )

In [7]:
configs = os.listdir('./configs')
for config in tqdm(configs):
    if config in ['.ipynb_checkpoints', 'base.yaml']:
        continue
    start_train(config)

  0%|          | 0/18 [00:00<?, ?it/s]

Loading data from ./data/ml-1m.txt...
Starting QAT with APOT...
Running dummy pass to initialize quantization parameters...
[QAT apot] Epoch 1/20 | Loss: 6.2927 | NDCG@10: 0.0006 | Hit@10: 0.0015
Saved best QAT checkpoint (NDCG: 0.0006)
[QAT apot] Epoch 2/20 | Loss: 5.6384 | NDCG@10: 0.0009 | Hit@10: 0.0025
Saved best QAT checkpoint (NDCG: 0.0009)
[QAT apot] Epoch 3/20 | Loss: 4.2597 | NDCG@10: 0.0012 | Hit@10: 0.0031
Saved best QAT checkpoint (NDCG: 0.0012)
[QAT apot] Epoch 4/20 | Loss: 3.0230 | NDCG@10: 0.0016 | Hit@10: 0.0040
Saved best QAT checkpoint (NDCG: 0.0016)
[QAT apot] Epoch 5/20 | Loss: 2.1825 | NDCG@10: 0.0017 | Hit@10: 0.0035
Saved best QAT checkpoint (NDCG: 0.0017)
[QAT apot] Epoch 6/20 | Loss: 1.6791 | NDCG@10: 0.0021 | Hit@10: 0.0043
Saved best QAT checkpoint (NDCG: 0.0021)
[QAT apot] Epoch 7/20 | Loss: 1.3722 | NDCG@10: 0.0022 | Hit@10: 0.0045
Saved best QAT checkpoint (NDCG: 0.0022)
[QAT apot] Epoch 8/20 | Loss: 1.1984 | NDCG@10: 0.0022 | Hit@10: 0.0040
[QAT apot] Ep

  6%|▌         | 1/18 [08:56<2:32:00, 536.49s/it]

Training curves saved to results/sasrec_apot_training_curves.png
Loading data from ./data/ml-1m.txt...
Starting AdaRound PTQ...
Loading FP32 checkpoint: checkpoints/sasrec_runs/sasrec_fp32/sasrec_fp32.pth
Running AdaRound calibration...
Validating AdaRound model on validation split...
[AdaRound][Val] NDCG@10: 0.0128 | Hit@10: 0.0273
Running final test evaluation for AdaRound model (user_test split)...


 22%|██▏       | 4/18 [10:03<27:54, 119.62s/it]  

...

Training curves saved to results/sasrec_qdrop_highp_training_curves.png
Loading data from ./data/ml-1m.txt...
Starting QAT with LSQ...
Running dummy pass to initialize quantization parameters...
[QAT lsq] Epoch 1/20 | Loss: 3.8632 | NDCG@10: 0.0008 | Hit@10: 0.0022
Saved best QAT checkpoint (NDCG: 0.0008)
[QAT lsq] Epoch 2/20 | Loss: 3.6093 | NDCG@10: 0.0006 | Hit@10: 0.0017
[QAT lsq] Epoch 3/20 | Loss: 3.4789 | NDCG@10: 0.0006 | Hit@10: 0.0015
[QAT lsq] Epoch 4/20 | Loss: 3.2796 | NDCG@10: 0.0006 | Hit@10: 0.0013
[QAT lsq] Epoch 5/20 | Loss: 3.0913 | NDCG@10: 0.0006 | Hit@10: 0.0015
[QAT lsq] Epoch 6/20 | Loss: 2.8814 | NDCG@10: 0.0008 | Hit@10: 0.0022
Saved best QAT checkpoint (NDCG: 0.0008)
[QAT lsq] Epoch 7/20 | Loss: 2.6846 | NDCG@10: 0.0008 | Hit@10: 0.0023
[QAT lsq] Epoch 8/20 | Loss: 2.4949 | NDCG@10: 0.0011 | Hit@10: 0.0030
Saved best QAT checkpoint (NDCG: 0.0011)
[QAT lsq] Epoch 9/20 | Loss: 2.3264 | NDCG@10: 0.0012 | Hit@10: 0.0033
Saved best QAT checkpoint (NDCG: 0.0012)
[Q

 94%|█████████▍| 17/18 [1:31:57<06:44, 404.66s/it]

Training curves saved to results/sasrec_lsq_4bit_asym_training_curves.png
Loading data from ./data/ml-1m.txt...
Starting QAT with QDROP...
Running dummy pass to initialize quantization parameters...
[QAT qdrop] Epoch 1/20 | Loss: 6.2848 | NDCG@10: 0.0006 | Hit@10: 0.0015
Saved best QAT checkpoint (NDCG: 0.0006)
[QAT qdrop] Epoch 2/20 | Loss: 5.6310 | NDCG@10: 0.0009 | Hit@10: 0.0025
Saved best QAT checkpoint (NDCG: 0.0009)
[QAT qdrop] Epoch 3/20 | Loss: 4.2076 | NDCG@10: 0.0012 | Hit@10: 0.0028
Saved best QAT checkpoint (NDCG: 0.0012)
[QAT qdrop] Epoch 4/20 | Loss: 2.9528 | NDCG@10: 0.0017 | Hit@10: 0.0038
Saved best QAT checkpoint (NDCG: 0.0017)
[QAT qdrop] Epoch 5/20 | Loss: 2.1265 | NDCG@10: 0.0020 | Hit@10: 0.0041
Saved best QAT checkpoint (NDCG: 0.0020)
[QAT qdrop] Epoch 6/20 | Loss: 1.6299 | NDCG@10: 0.0020 | Hit@10: 0.0040
Saved best QAT checkpoint (NDCG: 0.0020)
[QAT qdrop] Epoch 7/20 | Loss: 1.3350 | NDCG@10: 0.0024 | Hit@10: 0.0045
Saved best QAT checkpoint (NDCG: 0.0024)
[QA

100%|██████████| 18/18 [1:41:36<00:00, 338.68s/it]

Training curves saved to results/sasrec_qdrop_4bit_training_curves.png


In [9]:
ls checkpoints/sasrec_runs

sasrec_adaround/                  sasrec_fp32_low_dropout/
sasrec_adaround_4bit/             sasrec_fp32_low_dropout__weights/
sasrec_adaround_4bit__weights/    sasrec_lsq/
sasrec_adaround__weights/         sasrec_lsq_4bit_asym/
sasrec_adaround_v2/               sasrec_lsq_4bit_asym__weights/
sasrec_adaround_v2__weights/      sasrec_lsq__weights/
sasrec_apot/                      sasrec_lsq_asym/
sasrec_apot_4bit/                 sasrec_lsq_asym__weights/
sasrec_apot_4bit__weights/        sasrec_lsq_run/
sasrec_apot__weights/             sasrec_lsq_run__weights/
sasrec_apot_perchannel/           sasrec_qdrop/
sasrec_apot_perchannel__weights/  sasrec_qdrop_4bit/
sasrec_fp32/                      sasrec_qdrop_4bit__weights/
sasrec_fp32__weights/             sasrec_qdrop__weights/
sasrec_fp32_large/                sasrec_qdrop_highp/
sasrec_fp32_large__weights/       sasrec_qdrop_highp__weights/
